In [1]:
import pandas as pd
import json
import gzip
from typing import Any

## Load Data

In [2]:
CATEGORIES = ['All_Beauty', 'Amazon_Fashion']
MAX_REVIEWS_PER_CATEGORY = 999999999999
SEED = 67

def stream_jsonl(path: str, fields: list[str] | None = None, limit: int | None = None):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit is not None and i >= limit:
                break
            obj = json.loads(line)
            if fields is not None:
                obj = {k: obj.get(k) for k in fields}
            yield obj

def load_items(categories: list[str], parent_asins: list[str] | None = None):
    df_items = pd.DataFrame()
    for category in categories:
        print(f"Loading items for category: {category}")
        items_category: list[dict[Any, Any]] = []
        for item in stream_jsonl(f"../data/meta_{category}.jsonl.gz", limit=None):
            if parent_asins is not None and item.get('parent_asin') not in parent_asins:
                continue
            items_category.append(item)

        df_items = pd.concat([df_items, pd.DataFrame(items_category)], ignore_index=True)
    return df_items

def load_reviews(categories: list[str]):
    df_reviews = pd.DataFrame()
    for category in categories:
        print(f"Loading reviews for category: {category}")
        reviews = list(stream_jsonl(f"../data/{category}.jsonl.gz", limit=MAX_REVIEWS_PER_CATEGORY))
        df_reviews_category = pd.DataFrame(reviews)
        df_reviews_category['category'] = category
        df_reviews = pd.concat([df_reviews, df_reviews_category], ignore_index=True)
    return df_reviews

df_reviews = load_reviews(CATEGORIES)
df_items = load_items(CATEGORIES, parent_asins=df_reviews['asin'].unique().tolist())

Loading reviews for category: All_Beauty
Loading reviews for category: Amazon_Fashion
Loading items for category: All_Beauty
Loading items for category: Amazon_Fashion


In [5]:
display(df_reviews.head())
df_reviews.info()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,category
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True,All_Beauty
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True,All_Beauty
2,5.0,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True,All_Beauty
3,1.0,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True,All_Beauty
4,5.0,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True,All_Beauty


<class 'pandas.DataFrame'>
RangeIndex: 3202467 entries, 0 to 3202466
Data columns (total 11 columns):
 #   Column             Dtype  
---  ------             -----  
 0   rating             float64
 1   title              str    
 2   text               str    
 3   images             object 
 4   asin               str    
 5   parent_asin        str    
 6   user_id            str    
 7   timestamp          int64  
 8   helpful_vote       int64  
 9   verified_purchase  bool   
 10  category           str    
dtypes: bool(1), float64(1), int64(2), object(1), str(6)
memory usage: 953.9+ MB


In [6]:
display(df_items.head())
df_items.info()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Howard Products,[],{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,None
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Yes To,[],"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM,None
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Levine Health Products,[],{'Manufacturer': 'Levine Health Products'},B000B658RI,None
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Cherioll,[],"{'Brand': 'Cherioll', 'Item Form': 'Powder', '...",B088FKY3VD,None
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],Precision,[],{'UPC': '644287689178'},B07NGFDN6G,None


<class 'pandas.DataFrame'>
RangeIndex: 921435 entries, 0 to 921434
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   main_category    921435 non-null  str    
 1   title            921435 non-null  str    
 2   average_rating   921435 non-null  float64
 3   rating_number    921435 non-null  int64  
 4   features         921435 non-null  object 
 5   description      921435 non-null  object 
 6   price            52648 non-null   float64
 7   images           921435 non-null  object 
 8   videos           921435 non-null  object 
 9   store            883440 non-null  str    
 10  categories       921435 non-null  object 
 11  details          921435 non-null  object 
 12  parent_asin      921435 non-null  str    
 13  bought_together  0 non-null       object 
dtypes: float64(2), int64(1), object(7), str(4)
memory usage: 210.5+ MB


## Explore Data

### Data field availability and quality

Check whether item metadata and review fields are present and usable for downstream recommender features.

In [7]:
def is_missing(value: Any) -> bool:
    if value is None or value is pd.NA:
        return True
    if isinstance(value, float) and pd.isna(value):
        return True
    return False


def is_non_empty_value(value: Any) -> bool:
    if is_missing(value):
        return False
    if isinstance(value, str):
        return bool(value.strip())
    if isinstance(value, dict):
        return len(value) > 0 and any(
            is_non_empty_value(key) or is_non_empty_value(val)
            for key, val in value.items()
        )
    if isinstance(value, (list, tuple, set)):
        return len(value) > 0 and any(is_non_empty_value(item) for item in value)
    return True


def compact_example(value: Any, max_chars: int = 160) -> str:
    text = repr(value)
    return text if len(text) <= max_chars else text[: max_chars - 3] + "..."


def sample_non_empty_values(series: pd.Series, n: int = 3) -> list[str]:
    values = []
    for value in series:
        if is_non_empty_value(value):
            values.append(compact_example(value))
        if len(values) == n:
            break
    return values


def field_availability(df: pd.DataFrame, field_specs: list[tuple[str, str]]) -> pd.DataFrame:
    rows = []
    total_rows = len(df)
    for label, column in field_specs:
        present = column in df.columns
        if present:
            series = df[column]
            non_null = int(series.notna().sum())
            non_empty_mask = series.map(is_non_empty_value)
            non_empty = int(non_empty_mask.sum())
            examples = sample_non_empty_values(series)
        else:
            non_null = 0
            non_empty = 0
            examples = []

        rows.append({
            "field": label,
            "source_column": column,
            "present": present,
            "rows": total_rows,
            "non_null": non_null,
            "non_null_pct": round(100 * non_null / total_rows, 2) if total_rows else 0,
            "non_empty": non_empty,
            "non_empty_pct": round(100 * non_empty / total_rows, 2) if total_rows else 0,
            "examples": examples,
        })
    return pd.DataFrame(rows)

#### Availability summary

In [8]:
ITEM_FIELD_SPECS = [
    ("Item descriptions", "description"),
    ("Item features", "features"),
    ("Item prices", "price"),
    ("Item images", "images"),
    ("Item videos", "videos"),
    ("Item categories", "categories"),
    ("Item details", "details"),
    ("Item bought together", "bought_together"),
]

REVIEW_FIELD_SPECS = [
    ("Review verified purchase", "verified_purchase"),
    ("Review helpful vote", "helpful_vote"),
]

item_field_quality = field_availability(df_items, ITEM_FIELD_SPECS)
review_field_quality = field_availability(df_reviews, REVIEW_FIELD_SPECS)

display(item_field_quality)
display(review_field_quality)

,field,source_column,present,rows,non_null,non_null_pct,non_empty,non_empty_pct,examples
0,Item descriptions,description,True,921435,921435,100.00,68665,7.45,[['The Precision Plunger Bars are designed to ...
1,Item features,features,True,921435,921435,100.00,464336,50.39,"[['Material: 304 Stainless Steel; Brass tip', ..."
2,Item prices,price,True,921435,52648,5.71,52648,5.71,"[6.99, 86.95, 79.5]"
3,Item images,images,True,921435,921435,100.00,921434,100.00,[[{'thumb': 'https://m.media-amazon.com/images...
4,Item videos,videos,True,921435,921435,100.00,63946,6.94,"[[{'title': '2PC rainbow cloud bath bombs,Floa..."
5,Item categories,categories,True,921435,921435,100.00,0,0.00,[]
6,Item details,details,True,921435,921435,100.00,890486,96.64,[{'Package Dimensions': '7.1 x 5.5 x 3 inches;...
7,Item bought together,bought_together,True,921435,0,0.00,0,0.00,[]


,field,source_column,present,rows,non_null,non_null_pct,non_empty,non_empty_pct,examples
0,Review verified purchase,verified_purchase,True,3202467,3202467,100.0,3202467,100.0,"[True, True, True]"
1,Review helpful vote,helpful_vote,True,3202467,3202467,100.0,3202467,100.0,"[0, 1, 2]"


#### Field-specific quality checks

In [9]:
def list_field_quality(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    total_rows = len(df)
    for column in columns:
        lengths = df[column].map(lambda value: len(value) if isinstance(value, list) else 0)
        rows.append({
            "field": column,
            "list_values": int(df[column].map(lambda value: isinstance(value, list)).sum()),
            "non_empty_lists": int((lengths > 0).sum()),
            "non_empty_list_pct": round(100 * (lengths > 0).sum() / total_rows, 2) if total_rows else 0,
            "avg_list_length": round(lengths.mean(), 2) if total_rows else 0,
            "max_list_length": int(lengths.max()) if total_rows else 0,
        })
    return pd.DataFrame(rows)


price_numeric = pd.to_numeric(df_items["price"], errors="coerce")
price_quality = pd.DataFrame([{
    "rows": len(df_items),
    "numeric_values": int(price_numeric.notna().sum()),
    "numeric_pct": round(100 * price_numeric.notna().mean(), 2),
    "positive_prices": int((price_numeric > 0).sum()),
    "positive_price_pct": round(100 * (price_numeric > 0).mean(), 2),
    "min": price_numeric.min(),
    "median": price_numeric.median(),
    "mean": round(price_numeric.mean(), 2),
    "max": price_numeric.max(),
}])

list_quality = list_field_quality(
    df_items,
    ["description", "features", "images", "videos", "categories"],
)

detail_lengths = df_items["details"].map(lambda value: len(value) if isinstance(value, dict) else 0)
details_quality = pd.DataFrame([{
    "rows": len(df_items),
    "dict_values": int(df_items["details"].map(lambda value: isinstance(value, dict)).sum()),
    "non_empty_dicts": int((detail_lengths > 0).sum()),
    "non_empty_dict_pct": round(100 * (detail_lengths > 0).mean(), 2),
    "avg_key_count": round(detail_lengths.mean(), 2),
    "max_key_count": int(detail_lengths.max()),
}])

detail_key_counts: dict[str, int] = {}
for details in df_items["details"]:
    if isinstance(details, dict):
        for key in details.keys():
            detail_key_counts[key] = detail_key_counts.get(key, 0) + 1

top_detail_keys = (
    pd.DataFrame(
        [{"detail_key": key, "count": count} for key, count in detail_key_counts.items()]
    )
    .sort_values("count", ascending=False)
    .head(20)
    .reset_index(drop=True)
)

bought_together_non_empty = df_items["bought_together"].map(is_non_empty_value)
bought_together_quality = pd.DataFrame([{
    "rows": len(df_items),
    "non_null": int(df_items["bought_together"].notna().sum()),
    "non_empty": int(bought_together_non_empty.sum()),
    "non_empty_pct": round(100 * bought_together_non_empty.mean(), 2),
}])

verified_purchase_quality = (
    df_reviews["verified_purchase"]
    .value_counts(dropna=False)
    .rename_axis("verified_purchase")
    .reset_index(name="count")
)
verified_purchase_quality["pct"] = round(
    100 * verified_purchase_quality["count"] / len(df_reviews), 2
)

helpful_vote_numeric = pd.to_numeric(df_reviews["helpful_vote"], errors="coerce")
helpful_vote_quality = pd.DataFrame([{
    "rows": len(df_reviews),
    "numeric_values": int(helpful_vote_numeric.notna().sum()),
    "numeric_pct": round(100 * helpful_vote_numeric.notna().mean(), 2),
    "null_values": int(helpful_vote_numeric.isna().sum()),
    "zero_votes": int((helpful_vote_numeric == 0).sum()),
    "positive_votes": int((helpful_vote_numeric > 0).sum()),
    "positive_vote_pct": round(100 * (helpful_vote_numeric > 0).mean(), 2),
    "min": helpful_vote_numeric.min(),
    "median": helpful_vote_numeric.median(),
    "mean": round(helpful_vote_numeric.mean(), 2),
    "max": helpful_vote_numeric.max(),
}])

display(price_quality)
display(list_quality)
display(details_quality)
display(top_detail_keys)
display(bought_together_quality)
display(verified_purchase_quality)
display(helpful_vote_quality)

,rows,numeric_values,numeric_pct,positive_prices,positive_price_pct,min,median,mean,max
0,921435,52648,5.71,52648,5.71,0.01,18.95,38.63,13000.0


,field,list_values,non_empty_lists,non_empty_list_pct,avg_list_length,max_list_length
0,description,921435,68665,7.45,0.18,152
1,features,921435,464336,50.39,1.08,26
2,images,921435,921434,100.00,4.59,31
3,videos,921435,63946,6.94,0.14,10
4,categories,921435,0,0.00,0.00,0


,rows,dict_values,non_empty_dicts,non_empty_dict_pct,avg_key_count,max_key_count
0,921435,921435,890486,96.64,3.41,24


,detail_key,count
0,Date First Available,782218
1,Package Dimensions,592835
2,Item model number,384620
3,Is Discontinued By Manufacturer,332097
4,Product Dimensions,181756
5,Brand,111103
6,Department,108066
7,Manufacturer,71060
8,UPC,59397
9,Material,51720


,rows,non_null,non_empty,non_empty_pct
0,921435,0,0,0.0


,verified_purchase,count,pct
0,True,2972671,92.82
1,False,229796,7.18


,rows,numeric_values,numeric_pct,null_values,zero_votes,positive_votes,positive_vote_pct,min,median,mean,max
0,3202467,3202467,100.0,0,2520276,682191,21.3,0,0.0,0.64,954


#### Populated field samples

In [ ]:
pd.set_option("display.max_colwidth", None)
for label, column in ITEM_FIELD_SPECS:
    mask = df_items[column].map(is_non_empty_value)
    sample_columns = ["parent_asin", column]
    print(f"{label}: {int(mask.sum()):,} populated rows")
    display(df_items.loc[mask, sample_columns].head(3))

for label, column in REVIEW_FIELD_SPECS:
    mask = df_reviews[column].map(is_non_empty_value)
    sample_columns = ["asin", "parent_asin", "user_id", column]
    print(f"{label}: {int(mask.sum()):,} populated rows")
    display(df_reviews.loc[mask, sample_columns].head(3))

Item descriptions: 68,665 populated rows


,parent_asin,description
4,B07NGFDN6G,"[The Precision Plunger Bars are designed to work seamlessly with the Precision Disposable 1. 25"" Contoured Soft Cartridge Grips and the Precision Disposable 1"" Textured Soft Cartridge Grips to drive cartridge needles with vice style or standard tattoo machine setups. These plunger bars are manufactured from 304 Stainless Steel and feature a brass tip. The plungers are sold in a bag of ten in your choice of 88mm, 93mm, or 98mm length.]"
5,B07G9GWFSM,"[Description, The false toenails are durable with perfect length. You have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like. Plus, ABS is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own toenails. Fit well to your natural toenails. Non toxic, no smell, no harm to your health., Feature, - Color: As Shown.- Material: ABS.- Size: 14.3 x 7.2 x 1cm., Package Including, 100 x Pieces fake toenails]"
8,B01ERJEGS6,[Edt spray 3 oz design house: balmain]


Item features: 464,336 populated rows


,parent_asin,features
4,B07NGFDN6G,"[Material: 304 Stainless Steel; Brass tip, Lengths Available: 88mm, 93mm, 98mm, Accepts cartridge needles with vice style tattoo machines, Works perfectly with Precision Disposable Soft Cartridge Grips, Price per one bag of 10 plungers]"
5,B07G9GWFSM,"[The false toenails are durable with perfect length. You have the option to wear them long or clip them short, easy to trim and file them to in any length and shape you like., ABS is kind of green enviromental material, and makes the nails durable, breathable, light even no pressure on your own nails., Fit well to your natural toenails. Non toxic, no smell, no harm to your health., Wonderful as gift for girlfriend, family and friends., The easiest and most efficient way to do your toenail tips for manicures or nail art designs. It's fashion, creative, a useful accessory brighten up your look, also as a gift.]"
8,B01ERJEGS6,[Extatic Balmain Gold Musk By Balmain Edt Spray 3 Oz]


Item prices: 52,648 populated rows


...

,parent_asin,images
0,B01CUPMQZE,"[{'thumb': 'https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg', 'variant': 'MAIN', 'hi_res': None}, {'thumb': 'https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41w2yznfuZL.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg'}]"
1,B076WQZGPM,"[{'thumb': 'https://m.media-amazon.com/images/I/41b+11d5igL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41b+11d5igL.jpg', 'variant': 'MAIN', 'hi_res': 'https://m.media-amazon.com/images/I/71g1lP0pMbL._SL1500_.jpg'}, {'thumb': 'https://m.media-amazon.com/images/I/41j2ocUzCtL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41j2ocUzCtL.jpg', 'variant': 'PT01', 'hi_res': 'https://m.media-amazon.com/images/I/81OqvR94isL._SL1500_.jpg'}]"
2,B000B658RI,"[{'thumb': 'https://m.media-amazon.com/images/I/31bz+uqzWCL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/31bz+uqzWCL.jpg', 'variant': 'MAIN', 'hi_res': None}, {'thumb': 'https://m.media-amazon.com/images/I/31bz+uqzWCL._SS40_.jpg', 'large': 'https://m.media-amazon.com/images/I/31bz+uqzWCL.jpg', 'variant': 'PT01', 'hi_res': None}]"


Item videos: 63,946 populated rows


,parent_asin,videos
19,B07R13ZSV5,"[{'title': '2PC rainbow cloud bath bombs,Float on Water&Release Vivid Rainbow Color', 'url': 'https://www.amazon.com/vdp/518d95dae7fa43e8a78cbc009290c421?ref=dp_vse_rvc_0', 'user_id': ''}]"
21,B07X1TK3VS,"[{'title': 'VIROCHEMISTRY Pheromone Perfume To Attract Men', 'url': 'https://www.amazon.com/vdp/dc7126b4c6ca4db1b60de56c02a84e00?ref=dp_vse_rvc_0', 'user_id': ''}, {'title': 'Watch Before You Buy Raw & AlphaMale Pheromone Cologne', 'url': 'https://www.amazon.com/vdp/0a2336ec1cc4445a8296af399c0d52ea?ref=dp_vse_rvc_1', 'user_id': '/shop/srkent1989'}, {'title': 'Nontoxic cologne smell', 'url': 'https://www.amazon.com/vdp/07bea2cfbcbe41339a2e74101f4fbae9?ref=dp_vse_rvc_2', 'user_id': '/shop/kelsiecrawley'}]"
41,B08LYT4Q2X,"[{'title': 'Organic Sweet Almond Oil and Coconut Oil', 'url': 'https://www.amazon.com/vdp/78bb9dafd25743beab5b4c4703660a38?ref=dp_vse_rvc_0', 'user_id': ''}, {'title': 'Organic Natural Moisturizing Sweet Almond Oil by Shiny Leaf', 'url': 'https://www.amazon.com/vdp/0ee4db59aaec47939c526596e0ae77e0?ref=dp_vse_rvc_1', 'user_id': ''}, {'title': 'Rehydrate natural hair #haircare #naturalhair', 'url': 'https://www.amazon.com/vdp/095965c4c0ea4f7e81ac3f0a0d1163da?ref=dp_vse_rvc_2', 'user_id': '/shop/influencer-99029ed3'}, {'title': 'Silky Smooth Coconutty Wonder Oil With So Many Great Uses', 'url': 'https://www.amazon.com/vdp/0dfa5ea42fac45b7978bc8ae0ec7eb8e?ref=dp_vse_rvc_3', 'user_id': '/shop/startanonlinebusiness'}, {'title': 'NOW Solutions, Sweet Almond Oil Review!', 'url': 'https://www.amazon.com/vdp/09bd904bc8eb4c70b5863f3c733e627a?ref=dp_vse_rvc_4', 'user_id': '/shop/influencer-98d451b5'}, {'title': 'I use this after waxing, as a hair oil, for derma-planing ', 'url': 'https://www.amazon.com/vdp/0bdb5b7d51c341fb9a982dda2bd3d5af?ref=dp_vse_rvc_5', 'user_id': '/shop/influencer-74e6ffad'}, {'title': 'Body care routine #skincare #beauty #bodycare ', 'url': 'https://www.amazon.com/vdp/06f18c004866470f8aecdf2c2237ce4f?ref=dp_vse_rvc_6', 'user_id': '/shop/influencer-99029ed3'}, {'title': 'Pura d'or mct fractionated coconut oil review ', 'url': 'https://www.amazon.com/vdp/03b0a7c0592c4004adaaea0974830c3c?ref=dp_vse_rvc_7', 'user_id': '/shop/hlscrystalshop'}, {'title': 'Sweet Almond Oil Review', 'url': 'https://www.amazon.com/vdp/0ddb5898075d477aaf193b2a98c1ddfc?ref=dp_vse_rvc_8', 'user_id': '/shop/lacouflipse'}, {'title': 'Our Point of View on PURA D'OR Organic Sweet Almond Oil', 'url': 'https://www.amazon.com/vdp/07ca56d8eb364fea876a9591ba1c1810?ref=dp_vse_rvc_9', 'user_id': '/shop/influencer-20a38664'}]"


Item categories: 0 populated rows


,parent_asin,categories
